# Day 3 Session 2 -- Deployment Exercises

In the demos you saw how to build a validated AI service, cache responses semantically, monitor requests with structured logging, route queries by complexity, and track costs per model. Now you will combine the exact-match and semantic-match caching patterns into a single multi-tier caching system -- a critical production component that cuts LLM costs by serving repeated and similar queries from cache.

In [1]:
!pip install -q openai chromadb python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


---
## Setup

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = os.getenv("LANGCHAIN_TRACING_V2", "false")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT", "mckinsey-genai-3day")

import openai
import hashlib
import numpy as np
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

print("All imports successful!")

All imports successful!


---
## Recap

The AI Service demo showed request validation and structured responses. Semantic Caching demonstrated that embedding-based similarity can match queries that differ in phrasing but carry the same intent (threshold ~0.90-0.92). Monitoring and Cost Tracking gave us the instrumentation to measure cache effectiveness in production. Now you will build a two-tier cache that combines the speed of exact-match hashing with the flexibility of semantic matching.

---
## Task 3: Build a Multi-Tier Caching System

Build a `TieredCache` class with two cache tiers plus an LLM fallback:

1. **Tier 1 -- Exact match (hash-based, O(1)):** Normalize the prompt (lowercase, strip whitespace), compute its MD5 hash, and look up a dict. Instant hit for repeated identical queries.
2. **Tier 2 -- Semantic match (embedding-based):** Embed the prompt and compare cosine similarity against all cached embeddings. If the best match exceeds `semantic_threshold`, return the cached response.
3. **Tier 3 -- LLM call (cache miss):** Call the LLM, then store the response in both the exact-match dict and the semantic cache list.

**Requirements:**
- `query(prompt)` returns a dict with keys: `content`, `tier` (`"exact"` / `"semantic"` / `"llm"`), and `cached` (bool)
- Track hit counts per tier in a `stats` dict: `{"exact_hits": 0, "semantic_hits": 0, "misses": 0}`
- When a semantic hit occurs, also add it to the exact cache for future O(1) lookups
- Do NOT cache error responses

In [3]:
# Task 3 -- Build a Multi-Tier Caching System

class TieredCache:
    def __init__(self, semantic_threshold=0.92):
        self.exact_cache = {}          # hash -> response string
        self.semantic_cache = []       # list of (embedding, query, response)
        self.threshold = semantic_threshold
        self.embed_client = openai.OpenAI()
        self.llm = ChatOpenAI(
            model=os.getenv("OPENAI_MODEL_NAME", "gpt-4o-mini"),
            temperature=float(os.getenv("DEFAULT_TEMPERATURE", "0"))
        )
        self.stats = {"exact_hits": 0, "semantic_hits": 0, "misses": 0}

    def _hash(self, text):
        """Normalize and hash the prompt for exact-match lookup."""
        # TODO: return MD5 hex digest of text.strip().lower()
        pass

    def _embed(self, text):
        """Return the embedding vector for a single text string."""
        # TODO: call self.embed_client.embeddings.create(
        #           model=os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small"),
        #           input=[text]
        #       ).data[0].embedding
        pass

    def _cosine_sim(self, a, b):
        """Compute cosine similarity between two vectors."""
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

    def query(self, prompt):
        """Check exact cache -> semantic cache -> LLM. Return dict with content, tier, cached."""
        # TODO: Tier 1 -- compute hash, check self.exact_cache
        #       If hit: increment stats["exact_hits"], return {"content": ..., "tier": "exact", "cached": True}

        # TODO: Tier 2 -- embed prompt, loop over self.semantic_cache, find best cosine sim
        #       If best_sim >= self.threshold:
        #           increment stats["semantic_hits"]
        #           also store in exact_cache for future O(1) lookups
        #           return {"content": ..., "tier": "semantic", "similarity": best_sim, "cached": True}

        # TODO: Tier 3 -- call self.llm.invoke([...]).content
        #       increment stats["misses"]
        #       store in both exact_cache and semantic_cache
        #       return {"content": ..., "tier": "llm", "cached": False}
        pass


# --------------- Test ---------------
# cache = TieredCache(semantic_threshold=0.90)
#
# test_queries = [
#     "What are the key trends in healthcare M&A?",                             # LLM (miss)
#     "What are the key trends in healthcare M&A?",                             # Exact hit
#     "What is driving mergers and acquisitions in the healthcare sector?",     # Semantic hit
#     "What are the main cost reduction levers in automotive manufacturing?",   # LLM (miss)
#     "What are the main cost reduction levers in automotive manufacturing?",   # Exact hit
# ]
#
# for q in test_queries:
#     result = cache.query(q)
#     tier_info = f"tier={result['tier']}"
#     if 'similarity' in result:
#         tier_info += f" sim={result['similarity']:.3f}"
#     print(f"[{tier_info:25s}] {q[:50]:50s} | {result['content'][:50]}...")
#
# print(f"\nCache Stats: {cache.stats}")

---
## Expected Output

When you uncomment the test block above and run it, you should see output similar to:

```
[tier=llm                    ] What are the key trends in healthcare M&A?         | Key trends in healthcare M&A include: 1. Consol...
[tier=exact                  ] What are the key trends in healthcare M&A?         | Key trends in healthcare M&A include: 1. Consol...
[tier=semantic sim=0.922     ] What is driving mergers and acquisitions in the he  | Key trends in healthcare M&A include: 1. Consol...
[tier=llm                    ] What are the main cost reduction levers in automot  | In automotive manufacturing, key cost reduction ...
[tier=exact                  ] What are the main cost reduction levers in automot  | In automotive manufacturing, key cost reduction ...

Cache Stats: {'exact_hits': 2, 'semantic_hits': 1, 'misses': 2}
```

*The semantic similarity score may vary but should be above your threshold (0.90) for the healthcare M&A paraphrase. The exact content of LLM responses will differ between runs. The key verification points are: query 2 hits exact cache, query 3 hits semantic cache, and the stats show 2 exact hits, 1 semantic hit, 2 misses.*